#Install

In [ ]:
!pip install pymupdf
!pip install pytesseract
!pip install -U langchain-community

!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract

!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
# from read_pdf import read_pdf_and_clean
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/fix_systematic_review/After_filter_with_match_full_systematic_2.xlsx')

In [ ]:
df = df[df['Final_decision'] == 'No']

#Read file

In [ ]:
# Set the source directory and collect all PDF files
from pathlib import Path
path = '/content/drive/MyDrive/fix_systematic_review/papers_2'
source_dir = Path(path)
pdf_files = list(source_dir.glob('*.pdf'))

# Create a mapping between cleaned filenames (e.g., derived from DOI or title)
# and the original filename (without the .pdf extension)
cleaned_to_path = {}

for f in pdf_files:
    original_name = f.stem  # Get the filename without .pdf
    cleaned_name = (
        original_name
        .replace("_2F", "/")
        # .replace("_", " ")
        .replace(":", "")
        .replace("_28", "(")
        .replace("_29", ")")
        .replace("_", "\'")
        .lower()
    )
    cleaned_to_path[cleaned_name] = f.stem


In [ ]:
file_texts = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    match = str(row.get("Matching_Filename")).lower()
    name = row.get("Matching_Filename")
    if match in cleaned_to_path:
        file_path = f"{path}/{cleaned_to_path[match]}.pdf"
        # print(filepath)
        docs = read_pdf_and_clean(file_path)
        pages_text = "\n".join([doc.page_content for doc in docs])
        file_texts[name] = pages_text
    else:
        print(match)

In [ ]:
import pickle

with open("/content/drive/MyDrive/fix_systematic_review/134_texts.pkl", "rb") as f:
    file_texts = pickle.load(f)


#Run GPT Q1 and Q2

##Q1

In [ ]:
prompt_template = """
You are an expert in systematic reviews.
Text to evaluate:
{text}

Question:
Are there clear research questions?

Instructions:
- Answer "Yes" if the text contains explicit research questions OR if such questions can be clearly inferred from the context.
- Answer "No" only if there are no explicit or clearly implied research questions.

Answer: <Yes or No>

Explanation: <If a research question is present (explicitly or implicitly), write it explicitly or paraphrase it. If not, explain what the text might indicate instead.>
"""

In [ ]:

# Iterate with progress bar
client = OpenAI(api_key="api_key")

# df['Response_Q1'] = None
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['Response_Q1']):
        continue
    name = row.Matching_Filename

    text = file_texts[name]

    full_prompt = prompt_template.format(text = text)


    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=2048,
        top_p=0.45,
        messages=[
            {"role": "user", "content": full_prompt}
        ]
    )

    # Extract and store response
    content = response.choices[0].message.content
    print(content)
    df.at[index, 'Response_Q1'] = content

In [ ]:
df.to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/Q1.xlsx')

In [ ]:
df["Answering_Q1"] = df["Response_Q1"].str.extract(r'Answer:\s*(Yes|No)', expand=False)
df["Explanation_Q1"] = df["Response_Q1"].str.extract(r'Explanation:\s*(.*)', expand=False)

df

In [ ]:
df["Answering_Q1"].value_counts()

,count
Answering_Q1,
Yes,134


##Q2

In [ ]:
prompt = """
You are an expert in systematic reviews.

Text to evaluate:
{text}

Research questions or research objectives extracted:
{question}

Task:
Determine whether the collected data are sufficient to address the research questions.

Respond in the exact format:
Answer: <Yes or No>
Explanation: <One to three sentences explaining your reasoning>
"""


In [ ]:

# Iterate with progress bar
client = OpenAI(api_key="api_key")

df['Response_Q2'] = None
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['Response_Q2']):
        continue
    name = row.Matching_Filename

    text = file_texts[name]
    question = row.Response_Q1

    full_prompt = prompt.format(text = text,question = question)


    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=2048,
        top_p=0.45,
        messages=[
            {"role": "user", "content": full_prompt}
        ]
    )

    # Extract and store response
    content = response.choices[0].message.content
    print(content)
    df.at[index, 'Response_Q2'] = content

In [ ]:
df["Answering_Q2"] = df["Response_Q2"].str.extract(r'Answer:\s*(Yes|No)', expand=False)
df["Explanation_Q2"] = df["Response_Q2"].str.extract(r'Explanation:\s*(.*)', expand=False)

df

In [ ]:
df.Answering_Q2.value_counts()

,count
Answering_Q2,
Yes,134


In [ ]:
df.to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/Q2.xlsx')

#Run llama Q1 and Q2

In [ ]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/Q2.xlsx')

In [ ]:
import os
import requests
import json

# Store your API key as an environment variable for safety:
OPENROUTER_API_KEY = 'api_key'
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    # Optional but recommended: helps OpenRouter show where requests come from
    "HTTP-Referer": "https://your-site-or-none.example",
    "X-Title": "Your App Name",
}

def call_openai_new(prompt, model="meta-llama/llama-3.3-70b-instruct",
                temperature=0, top_p=0.45, max_tokens=2048, timeout=60):
    """
    Replacement function for OpenAI client -> uses OpenRouter API.

    Args:
        prompt (str): user input text
        model (str): model to use (default Gemini Flash Lite)
        temperature (float): sampling temperature
        top_p (float): nucleus sampling
        max_tokens (int): maximum output tokens
        timeout (int): request timeout in seconds

    Returns:
        str: model's response content or error message
    """
    try:
        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "top_p": top_p,
            "max_tokens": max_tokens,
        }

        resp = requests.post(OPENROUTER_URL, headers=HEADERS,
                             data=json.dumps(payload), timeout=timeout)
        resp.raise_for_status()  # raises HTTPError if status != 200
        data = resp.json()

        # OpenRouter returns a format similar to OpenAI
        return data["choices"][0]["message"]["content"].strip()

    except requests.HTTPError as e:
        # Try to parse error body for more details
        try:
            err_body = resp.json()
        except Exception:
            err_body = resp.text
        return f"Error {resp.status_code}: {err_body}"
    except Exception as e:
        return f"Error: {e}"


##Q1

In [ ]:
save_interval = 10
output_file = '/content/drive/MyDrive/fix_systematic_review/MMAT/Q1_Q2_llama.xlsx'
# df['Response_Q1_llama'] = None
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['Response_Q1_llama']):
        continue
    name = row.Matching_Filename

    text = file_texts[name]

    full_prompt = prompt_template.format(text = text)
    result = call_openai_new(full_prompt)



    print(result)
    print('---')
    print(row['Answering_Q1'])
    df.at[index, 'Response_Q1_llama'] = result
        # Save progress every 'save_interval' rows
    if index % save_interval == 0 and index > 0:
        df.to_excel(output_file, index=False)
        print(f"💾 Progress saved at row {index} to {output_file}")
# Final save after processing all rows
df.to_excel(output_file, index=False)
print(f"✅ Processing completed. Final file saved to {output_file}")

In [ ]:
df["Answering_Q1_llama"] = df["Response_Q1_llama"].str.extract(
    r'(?:Final decision:\s*)?(Yes|No)', expand=False
)
df.Answering_Q1_llama.value_counts()

,count
Answering_Q1_llama,
Yes,132
No,2


In [ ]:
import pandas as pd
output_file = '/content/drive/MyDrive/fix_systematic_review/MMAT/Q1_Q2_llama.xlsx'
df = pd.read_excel(output_file)

In [ ]:
df["type_agreement_Q1"] = (df["Answering_Q1"] == df["Answering_Q1_llama"]).astype(int)


In [ ]:

total_agreed = df["type_agreement_Q1"].sum()
print(f"Number of agreements: {total_agreed} out of {len(df)}")


Number of agreements: 132 out of 134


In [ ]:
# df[df['type_agreement_Q1']== 1].sample(20).to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/label/agreed_sample_q1.xlsx')

In [ ]:
# df[~df['jugment_q1'].isna()].to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/label/jugment_q1_label_for_cheak.xlsx')

###Jugment Q1  - Claude

In [ ]:
import os
import requests
import json

# Store your API key as an environment variable for safety:
OPENROUTER_API_KEY = 'api_key'
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    # Optional but recommended: helps OpenRouter show where requests come from
    "HTTP-Referer": "https://your-site-or-none.example",
    "X-Title": "Your App Name",
}

def call_openai_new(prompt, model="anthropic/claude-sonnet-4", #"google/gemini-2.5-flash-lite"
                temperature=0, top_p=0.45, max_tokens=2048, timeout=60):
    """
    Replacement function for OpenAI client -> uses OpenRouter API.

    Args:
        prompt (str): user input text
        model (str): model to use (default Gemini Flash Lite)
        temperature (float): sampling temperature
        top_p (float): nucleus sampling
        max_tokens (int): maximum output tokens
        timeout (int): request timeout in seconds

    Returns:
        str: model's response content or error message
    """
    try:
        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "top_p": top_p,
            "max_tokens": max_tokens,
        }

        resp = requests.post(OPENROUTER_URL, headers=HEADERS,
                             data=json.dumps(payload), timeout=timeout)
        resp.raise_for_status()  # raises HTTPError if status != 200
        data = resp.json()

        # OpenRouter returns a format similar to OpenAI
        return data["choices"][0]["message"]["content"].strip()

    except requests.HTTPError as e:
        # Try to parse error body for more details
        try:
            err_body = resp.json()
        except Exception:
            err_body = resp.text
        return f"Error {resp.status_code}: {err_body}"
    except Exception as e:
        return f"Error: {e}"


In [ ]:
prompt_template = """
You are an impartial judge in a systematic review evaluation task.

Text to evaluate:
{text}

Task:
Are there clear research questions?

Instructions:
- Answer "Yes" if the text contains explicit research questions OR if such questions can be clearly inferred from the context.
- Answer "No" only if there are no explicit or clearly implied research questions.

Model A's Answer:
{answer_a}

Model B's Answer:
{answer_b}

Your role:
- Carefully compare both answers against the provided text and the task.
- Decide which model's answer (A or B) is more accurate and justified.
- If both are equally valid, you may answer "Tie".

Respond in the exact format:
Decision: <A or B or Tie>
Reasoning: <One to three sentences explaining why you made this decision>
"""

In [ ]:

from tqdm import tqdm
# output_file = "/content/drive/MyDrive/fix_systematic_review/MMAT/Type_progress_jugment.xlsx"  # Path to save progress file
# Loop over DataFrame rows with tqdm progress bar
save_interval =10
text_col = "Matching_Filename"
df['jugment_q1'] =None

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    number_id = row.get(text_col, None) # Get filename for this row


    if row.get(f"type_agreement_Q1", None) == 1:
      continue

    gpt = row.get(f"Response_Q1", None)
    llama = row.get(f"Response_Q1_llama", None)

    prompt = prompt_template.format(text=file_texts[number_id],answer_a =gpt,answer_b = llama )
        # Send prompt to the model and get the result
    result = call_openai_new(prompt)
    df.at[i, f'jugment_q1'] = result

#         # Save progress every 'save_interval' rows
#     if i % save_interval == 0 and i > 0:
#         df.to_excel(output_file, index=False)
#         print(f"💾 Progress saved at row {i} to {output_file}")

# # Final save after processing all rows
# df.to_excel(output_file, index=False)
# print(f"✅ Processing completed. Final file saved to {output_file}")

Processing rows: 100%|██████████| 134/134 [00:07<00:00, 18.43it/s]


In [ ]:
df["decision_extracted"] = df["jugment_q1"].str.extract(r"Decision:\s*([A-Za-z])")

df['Q1_final'] = df['Answering_Q1']
# df.loc[df['type_agreement'] == 0, 'Type_final'] = df.loc[df['type_agreement'] == 0, 'Type_llama_only_response']
for i,row in df.iterrows():
  if row['type_agreement_Q1'] == 0:
    res = row['decision_extracted']
    if res == 'A':
      df.at[i, 'Q1_final'] = row['Answering_Q1']
    else:
      df.at[i, 'Q1_final'] = row['Answering_Q1_llama']


In [ ]:
df.Q1_final.value_counts()

,count
Q1_final,
Yes,134


In [ ]:
df.to_excel( '/content/drive/MyDrive/fix_systematic_review/MMAT/Q1_Q2_llama.xlsx')

##Q2

In [ ]:
save_interval = 10
output_file = '/content/drive/MyDrive/fix_systematic_review/MMAT/Q1_Q2_llama.xlsx'
df['Response_Q2_llama'] = None
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['Response_Q2_llama']):
        continue
    name = row.Matching_Filename

    text = file_texts[name]
    question = row.Response_Q1

    full_prompt = prompt.format(text = text,question = question)

    # full_prompt = prompt_template.format(text = text)
    result = call_openai_new(full_prompt)



    print(result)
    print('---')
    print(row['Response_Q2'])
    df.at[index, 'Response_Q2_llama'] = result
        # Save progress every 'save_interval' rows
    if index % save_interval == 0 and index > 0:
        df.to_excel(output_file, index=False)
        print(f"💾 Progress saved at row {index} to {output_file}")
# Final save after processing all rows
df.to_excel(output_file, index=False)
print(f"✅ Processing completed. Final file saved to {output_file}")

In [ ]:
import pandas as pd
df= pd.read_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/Q1_Q2_llama.xlsx')

In [ ]:
df["Answering_Q2_llama"] = df["Response_Q2_llama"].str.extract(r'Answer:\s*(Yes|No)', expand=False)


In [ ]:
df.Answering_Q2_llama.value_counts()

,count
Answering_Q2_llama,
Yes,112
No,22


In [ ]:
df["Q2_agreement"] = (df["Answering_Q2"] == df["Answering_Q2_llama"]).astype(int)


In [ ]:
df.Q2_agreement.value_counts()

,count
Q2_agreement,
1,112
0,22


###jugment Q2 - Claude

In [ ]:

import os
import requests
import json

# Store your API key as an environment variable for safety:
OPENROUTER_API_KEY = 'Api_key'
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    # Optional but recommended: helps OpenRouter show where requests come from
    "HTTP-Referer": "https://your-site-or-none.example",
    "X-Title": "Your App Name",
}

def call_openai_new(prompt, model="anthropic/claude-sonnet-4", #"google/gemini-2.5-flash-lite"
                temperature=0, top_p=0.45, max_tokens=2048, timeout=60):
    """
    Replacement function for OpenAI client -> uses OpenRouter API.

    Args:
        prompt (str): user input text
        model (str): model to use (default Gemini Flash Lite)
        temperature (float): sampling temperature
        top_p (float): nucleus sampling
        max_tokens (int): maximum output tokens
        timeout (int): request timeout in seconds

    Returns:
        str: model's response content or error message
    """
    try:
        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "top_p": top_p,
            "max_tokens": max_tokens,
        }

        resp = requests.post(OPENROUTER_URL, headers=HEADERS,
                             data=json.dumps(payload), timeout=timeout)
        resp.raise_for_status()  # raises HTTPError if status != 200
        data = resp.json()

        # OpenRouter returns a format similar to OpenAI
        return data["choices"][0]["message"]["content"].strip()

    except requests.HTTPError as e:
        # Try to parse error body for more details
        try:
            err_body = resp.json()
        except Exception:
            err_body = resp.text
        return f"Error {resp.status_code}: {err_body}"
    except Exception as e:
        return f"Error: {e}"


In [ ]:
prompt_template = """
You are an impartial judge evaluating systematic review assessments.

Text to evaluate:
{text}

Research questions or research objectives:
{research_questions}

Model A's Assessment:
{answer_a}

Model B's Assessment:
{answer_b}

Task:
Determine which model better evaluates whether the . Do the collected data allow to address the research questions?

Respond in the exact format:
Decision: <A or B or Tie>
Reasoning: <One to three sentences explaining why the chosen answer is more accurate>
"""


In [ ]:

from tqdm import tqdm
output_file = "/content/drive/MyDrive/fix_systematic_review/MMAT/q2_jugment_1.xlsx"  # Path to save progress file

# Loop over DataFrame rows with tqdm progress bar
save_interval =10
df['jugment_Q2'] = None
text_col = "Matching_Filename"

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    number_id = row.get(text_col, None) # Get filename for this row


    if row.get(f"Q2_agreement", None) == 1:
      continue

    gpt = row.get(f"Response_Q2", None)
    llama = row.get(f"Response_Q2_llama", None)
    question = row.Response_Q1
    q_llama = row.Response_Q1_llama

    prompt = prompt_template.format(text=file_texts[number_id],answer_a =gpt,answer_b = llama,research_questions = question)
        # Send prompt to the model and get the result
    result = call_openai_new(prompt)
    df.at[i, f'jugment_Q2'] = result

        # Save progress every 'save_interval' rows
    if i % save_interval == 0 and i > 0:
        df.to_excel(output_file, index=False)
        print(f"💾 Progress saved at row {i} to {output_file}")

# Final save after processing all rows
df.to_excel(output_file, index=False)
print(f"✅ Processing completed. Final file saved to {output_file}")

Processing rows:  38%|███▊      | 51/134 [00:33<00:58,  1.41it/s]

💾 Progress saved at row 50 to /content/drive/MyDrive/fix_systematic_review/MMAT/q2_jugment_1.xlsx


Processing rows:  83%|████████▎ | 111/134 [01:30<00:12,  1.86it/s]

💾 Progress saved at row 110 to /content/drive/MyDrive/fix_systematic_review/MMAT/q2_jugment_1.xlsx


Processing rows: 100%|██████████| 134/134 [01:42<00:00,  1.31it/s]

💾 Progress saved at row 130 to /content/drive/MyDrive/fix_systematic_review/MMAT/q2_jugment_1.xlsx
✅ Processing completed. Final file saved to /content/drive/MyDrive/fix_systematic_review/MMAT/q2_jugment_1.xlsx


In [ ]:

output_file = "/content/drive/MyDrive/fix_systematic_review/MMAT/Type_progress_jugment.xlsx"  # Path to save progress file
df= pd.read_excel(output_file)

In [ ]:
df[df['Q2_agreement'] ==1 ].sample(20).to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/label/agreed_sample_q2.xlsx')

In [ ]:
df[~df['jugment_Q2'].isna()].to_excel('/content/drive/MyDrive/fix_systematic_review/MMAT/label/jugment_q2_label_for_cheak_2.xlsx')

In [ ]:
df["decision_extracted_q2"] = df["jugment_Q2"].str.extract(r"Decision:\s*([A-Za-z])")


In [ ]:
df.decision_extracted_q2.value_counts()

,count
decision_extracted_q2,
B,22


In [ ]:
df.decision_extracted_q2.value_counts()

,count
decision_extracted_q2,
B,17
A,5


In [ ]:
df.decision_extracted_q2.value_counts()

,count
decision_extracted_q2,
B,12
A,10
